# Exploratory Data Analysis (EDA) for POS Myanmar ML Models

**Project:** AI-Powered POS & Inventory Management System  
**Purpose:** Explore historical sales data to understand patterns for ML modeling  
**Date:** April 24, 2026  
**Developer:** James (Dev Agent)

---

## Objectives

1. **Data Quality Check:** Assess completeness, missing values, outliers
2. **Temporal Patterns:** Identify trends, seasonality, and cycles in sales
3. **Product Analysis:** Understand best-sellers, sales velocity, inventory turnover
4. **Market Basket:** Discover product associations for recommendations
5. **Feature Engineering:** Create derived features for ML models

## 1. Setup and Data Loading

In [ ]:
# Import libraries
import sys
import os

# Add parent directory to path for imports
sys.path.append(os.path.dirname(os.getcwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# ML Service utilities
from utils.db_connection import test_connection, query_to_dataframe
from utils.data_extraction import (
    extract_sales_history,
    extract_daily_sales_aggregates,
    extract_product_sales_history,
    extract_transaction_items
)

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✓ Libraries imported successfully")

In [ ]:
# Test database connection
test_connection()

In [ ]:
# Extract data (last 180 days = 6 months)
print("Extracting data from database...\n")

sales_history = extract_sales_history(days=180, export_csv=True)
daily_sales = extract_daily_sales_aggregates(days=180, export_csv=True)
product_sales = extract_product_sales_history(days=180, export_csv=True)
transaction_items = extract_transaction_items(days=180, export_csv=True)

print("\n✓ Data extraction complete!")

## 2. Data Quality Assessment

In [ ]:
# Sales history overview
print("=" * 60)
print("SALES HISTORY - DATA QUALITY")
print("=" * 60)
print(f"\nShape: {sales_history.shape}")
print(f"\nData Types:\n{sales_history.dtypes}")
print(f"\nMissing Values:\n{sales_history.isnull().sum()}")
print(f"\nBasic Statistics:\n{sales_history.describe()}")

In [ ]:
# Daily sales overview
print("=" * 60)
print("DAILY SALES - DATA QUALITY")
print("=" * 60)
print(f"\nShape: {daily_sales.shape}")
print(f"\nDate range: {daily_sales['date'].min()} to {daily_sales['date'].max()}")
print(f"\nDays with data: {len(daily_sales)}")
print(f"\nMissing Values:\n{daily_sales.isnull().sum()}")

## 3. Temporal Analysis - Sales Trends

In [ ]:
# Plot daily sales trend
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Sales over time
axes[0].plot(daily_sales['date'], daily_sales['sales'], marker='o', linestyle='-', linewidth=1, markersize=3)
axes[0].set_title('Daily Sales Revenue Over Time', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Sales (MMK)')
axes[0].grid(True, alpha=0.3)

# Transactions over time
axes[1].plot(daily_sales['date'], daily_sales['transactions'], marker='o', linestyle='-', 
             linewidth=1, markersize=3, color='green')
axes[1].set_title('Daily Transaction Count Over Time', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Number of Transactions')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Average daily sales: {daily_sales['sales'].mean():,.2f} MMK")
print(f"Average daily transactions: {daily_sales['transactions'].mean():.1f}")

In [ ]:
# Weekly pattern analysis
weekly_avg = daily_sales.groupby('day_of_week')[['sales', 'transactions']].mean().reset_index()
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekly_avg['day_name'] = weekly_avg['day_of_week'].apply(lambda x: day_names[x])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sales by day of week
axes[0].bar(weekly_avg['day_name'], weekly_avg['sales'], color='steelblue')
axes[0].set_title('Average Sales by Day of Week', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Average Sales (MMK)')
axes[0].tick_params(axis='x', rotation=45)

# Transactions by day of week
axes[1].bar(weekly_avg['day_name'], weekly_avg['transactions'], color='green')
axes[1].set_title('Average Transactions by Day of Week', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Average Transactions')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\nWeekly Pattern Summary:")
print(weekly_avg[['day_name', 'sales', 'transactions']])

## 4. Product Analysis

In [ ]:
# Top products by revenue
product_summary = product_sales.groupby(['product_id', 'product_name', 'category']).agg({
    'units_sold': 'sum',
    'revenue': 'sum',
    'transactions': 'sum'
}).reset_index()

product_summary = product_summary.sort_values('revenue', ascending=False)

print("=" * 60)
print("TOP PRODUCTS BY REVENUE")
print("=" * 60)
print(product_summary.head(10))

In [ ]:
# Visualize top products
top_products = product_summary.head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Revenue
axes[0].barh(top_products['product_name'], top_products['revenue'], color='coral')
axes[0].set_title('Top 10 Products by Revenue', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Revenue (MMK)')
axes[0].invert_yaxis()

# Units sold
axes[1].barh(top_products['product_name'], top_products['units_sold'], color='purple')
axes[1].set_title('Top 10 Products by Units Sold', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Units Sold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 5. Market Basket Analysis - Product Associations

In [ ]:
# Most frequently bought together
from itertools import combinations

# Group products by transaction
baskets = transaction_items.groupby('sale_id')['product_name'].apply(list).reset_index()

# Find product pairs
product_pairs = []
for products in baskets['product_name']:
    if len(products) >= 2:
        for pair in combinations(sorted(products), 2):
            product_pairs.append(pair)

# Count pairs
pair_counts = pd.Series(product_pairs).value_counts().head(10)

print("=" * 60)
print("TOP 10 PRODUCT PAIRS (Frequently Bought Together)")
print("=" * 60)
for pair, count in pair_counts.items():
    print(f"{pair[0]} + {pair[1]}: {count} times")

## 6. Summary Statistics and Insights

In [ ]:
# Overall summary
print("=" * 60)
print("OVERALL BUSINESS SUMMARY (Last 180 Days)")
print("=" * 60)
print(f"\nTotal Sales: {sales_history['total_amount'].sum():,.2f} MMK")
print(f"Total Profit: {sales_history['profit'].sum():,.2f} MMK")
print(f"Profit Margin: {(sales_history['profit'].sum() / sales_history['total_amount'].sum() * 100):.1f}%")
print(f"\nTotal Transactions: {len(sales_history):,}")
print(f"Average Transaction Value: {sales_history['total_amount'].mean():,.2f} MMK")
print(f"Average Items per Transaction: {sales_history['items_count'].mean():.1f}")
print(f"\nUnique Products Sold: {transaction_items['product_id'].nunique()}")
print(f"Total Units Sold: {transaction_items['quantity'].sum():,}")

# Best and worst days
best_day = daily_sales.loc[daily_sales['sales'].idxmax()]
worst_day = daily_sales.loc[daily_sales['sales'].idxmin()]

print(f"\nBest Sales Day: {best_day['date'].date()} - {best_day['sales']:,.2f} MMK ({best_day['transactions']:.0f} txns)")
print(f"Worst Sales Day: {worst_day['date'].date()} - {worst_day['sales']:,.2f} MMK ({worst_day['transactions']:.0f} txns)")

## 7. Key Insights for ML Modeling

**Observations:**

1. **Temporal Patterns:**
   - Identify weekly patterns (weekend vs weekday sales)
   - Look for monthly seasonality (month-end effects)
   - Note any trends (growth/decline)

2. **Product Insights:**
   - Identify high-value vs high-volume products
   - Understand stock velocity for different categories
   - ABC analysis for inventory prioritization

3. **Market Basket:**
   - Product associations for recommendation system
   - Cross-selling opportunities

4. **Data Quality:**
   - Completeness of historical data
   - Outlier detection needs
   - Feature engineering opportunities

**Next Steps:**
- Build sales forecasting model (ARIMA/SARIMAX)
- Train inventory prediction model (Random Forest)
- Create product recommendation model (Apriori)